# Ejercicio 8 — Diseño $3^{3-1}$ fraccionado (generador módulo 3)

**Objetivo.** Construir una tercera fracción de un $3^3$ (9 corridas en vez de 27) usando
el generador $x_3 = x_1 + x_2 \pmod 3$ (teoría §8), ajustar el modelo de segundo orden
sin interacciones, localizar el óptimo, y entender qué se sacrifica (la capacidad de
estimar interacciones) a cambio del ahorro de corridas.

**Factores:**
- $A$ = Temperatura de reacción: 50 (−1), 60 (0), 70 °C (+1)
- $B$ = Relación molar metanol:aceite: 4 (−1), 6 (0), 8 (+1)
- $C$ = Concentración de catalizador (KOH): 0.5 (−1), 1.0 (0), 1.5 % (+1)

**Respuesta:** Conversión a biodiésel (%)

**Dataset:** `../../datos/biodiesel-3k3-fraccion.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

df = pd.read_csv('../../datos/biodiesel-3k3-fraccion.csv')
print(f'Corridas: {len(df)}  (fracción 3^(3-1) = 9;  3^3 completo = 27)')
print(df)

## 1. Verificación del generador de la fracción

La fracción se construye recorriendo las 9 combinaciones de $(x_1,x_2)$ y calculando
$x_3$ con el generador módulo 3 de la teoría (§8):

$$x_3^{\text{Yates}} = \bigl(x_1^{\text{Yates}} + x_2^{\text{Yates}}\bigr) \bmod 3$$

In [ ]:
x1_yates = df['x1'] + 1
x2_yates = df['x2'] + 1
x3_generado = (x1_yates + x2_yates) % 3 - 1

verificacion = pd.DataFrame({'x1': df['x1'], 'x2': df['x2'],
                             'x3_generado': x3_generado, 'x3_dataset': df['x3']})
print(verificacion)
print(f'\n¿x3 coincide con el generador C=A+B mod 3?  {(x3_generado.values == df["x3"].values).all()}')

print('\nOrtogonalidad de las columnas del diseño (deben dar 0):')
for a, b in [('x1', 'x2'), ('x1', 'x3'), ('x2', 'x3')]:
    print(f'  {a} · {b} = {np.dot(df[a], df[b])}')

## 2. Modelo de segundo orden (solo efectos principales L/Q)

Con 9 corridas solo alcanzan 2 gl de error para un modelo con 7 parámetros
($\beta_0$ + 3 lineales + 3 cuadráticos). Los términos de interacción quedan
**aliados** (confundidos) con los efectos principales por el propio generador y no se
incluyen en el modelo.

In [ ]:
modelo = smf.ols('conversion ~ x1 + x2 + x3 + I(x1**2) + I(x2**2) + I(x3**2)', data=df).fit()
print(modelo.summary())

In [ ]:
anova1 = sm.stats.anova_lm(modelo, typ=1)
print(anova1.round(4))

## 3. Punto óptimo (sin términos de interacción, la matriz $B$ es diagonal)

In [ ]:
p = modelo.params
b_vec = np.array([p['x1'], p['x2'], p['x3']])
B_mat = np.diag([p['I(x1 ** 2)'], p['I(x2 ** 2)'], p['I(x3 ** 2)']])
x_s = -np.linalg.solve(2*B_mat, b_vec)

y_s = modelo.predict(pd.DataFrame({'x1': [x_s[0]], 'x2': [x_s[1]], 'x3': [x_s[2]]}))[0]

centros_r = np.array([60, 6, 1.0])
deltas_r  = np.array([10, 2, 0.5])
x_real = centros_r + x_s * deltas_r

print('Punto estacionario (codificado):', x_s.round(3))
print(f'Conversión estimada: {y_s:.2f} %')
print('Óptimo real: temp=%.1f °C, ratio=%.2f, catalizador=%.2f %%' % tuple(x_real))

fuera = np.abs(x_s) > 1
print('¿Fuera de la región experimental (|x|>1)?', fuera.tolist())

## 4. Qué se sacrifica: el aliasing de las interacciones

Si se intenta ajustar el modelo completo de segundo orden (con las 3 interacciones de
dos factores), el modelo queda **sobre-parametrizado**: 10 parámetros para solo 9
corridas. La matriz de diseño pierde rango — no hay suficiente información para separar
las interacciones de los efectos principales.

In [ ]:
formula_full = ('conversion ~ x1+x2+x3+I(x1**2)+I(x2**2)+I(x3**2)'
                '+x1:x2+x1:x3+x2:x3')
modelo_full = smf.ols(formula_full, data=df).fit()
rank = np.linalg.matrix_rank(modelo_full.model.exog)
ncols = modelo_full.model.exog.shape[1]
print(f'Modelo con interacciones: rango={rank}, columnas={ncols}')
print('→ NO estimable: cada interacción está aliada con alguno de los efectos')
print('  principales por construcción del generador x3 = x1+x2 (mod 3).')

## 5. Costo-beneficio: fracción vs. $3^3$ completo

In [ ]:
resumen = pd.DataFrame({
    'Criterio': ['Corridas', 'Parámetros estimables (2° orden)',
                 'GL de error', '¿Interacciones estimables?'],
    'Fracción $3^{3-1}$': ['9', '7 (sin interac.)', '2', 'No'],
    '$3^3$ completo':      ['27', '10 (con interac.)', '17', 'Sí'],
})
print(resumen.to_string(index=False))
print('\nLa fracción cuesta 1/3 de las corridas del diseño completo. Es una buena')
print('elección cuando, por conocimiento previo del proceso, se puede asumir que las')
print('interacciones de dos factores son despreciables. Si esa suposición es falsa,')
print('las interacciones sesgarán las estimaciones de los efectos principales con los')
print('que están aliadas, sin que el analista pueda detectarlo con estos 9 datos.')

## 6. Conclusión

- El generador $x_3 = x_1+x_2 \pmod 3$ produce una tercera fracción (9 de 27 corridas)
  perfectamente ortogonal en $x_1$, $x_2$, $x_3$: los efectos principales L/Q se estiman
  sin sesgo, igual que en el $3^3$ completo.
- El ahorro (67% menos corridas) tiene un costo: las interacciones de dos factores quedan
  aliadas con los efectos principales y **no son estimables** con este diseño; intentar
  ajustarlas produce una matriz de diseño deficiente en rango.
- El punto óptimo hallado (temperatura ≈ 63 °C, relación molar ≈ 7.0, catalizador ≈
  1.15 %) cae dentro de la región experimental, por lo que la recomendación es una
  interpolación válida — sujeta a la suposición de que no hay interacciones relevantes.
- **Regla práctica.** Usa un $3^{k-p}$ fraccionado solo en la etapa exploratoria, cuando
  el screening previo ($2^{k-p}$) ya descartó interacciones importantes entre estos
  factores; si existe duda razonable sobre alguna interacción, corre el $3^k$ completo o
  un CCD/Box-Behnken que sí la estime.